# Notebook IV.5 — Corriente no sinusoidal, espectro y THD

Este notebook acompaña la **Ventana computacional IV.5** del Capítulo IV.

Aquí entramos por primera vez en un contexto explícitamente eléctrico.  
El propósito es mostrar cómo una **corriente periódica no sinusoidal** puede interpretarse como una **estructura armónica organizada**, susceptible de análisis, reconstrucción y medición mediante índices de distorsión como el **THD**.

Esta ventana constituye también la puerta de entrada al pequeño laboratorio armónico en sistemas eléctricos.

## Ejecutar este notebook en Google Colab

[![Run in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricomelgozajjesus/monografia-armonica/blob/main/python-lab/notebooks/Notebook_IV_05_THD_y_espectro.ipynb)

## 1. Idea general

En muchos problemas de calidad de la energía, la señal de corriente deja de ser casi sinusoidal.  
Esto puede deberse a la presencia de convertidores, rectificadores o cargas no lineales.

En este notebook construiremos primero una señal periódica distorsionada de manera controlada, para después:

1. visualizarla en el dominio del tiempo,
2. extraer su contenido armónico,
3. reconstruirla con distinto número de armónicos,
4. calcular su **THD**.

La idea es trabajar en un entorno sencillo pero conceptualmente cercano al análisis armónico de sistemas eléctricos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq

## 2. Definir una corriente periódica no sinusoidal

Construiremos una corriente con fundamental dominante y algunos armónicos impares.
Esto no pretende modelar todavía un dispositivo específico, sino ofrecer una estructura armónica clara para experimentación.

In [ ]:
f1 = 60.0                 # frecuencia fundamental [Hz]
w1 = 2*np.pi*f1
T1 = 1/f1

# Duración y muestreo
n_periods = 12
samples_per_period = 1200
N = n_periods * samples_per_period
t = np.linspace(0, n_periods*T1, N, endpoint=False)

# Armónicos prescritos
I1 = 100.0
I3 = 18.0
I5 = 9.0
I7 = 5.0
phi1 = 0.0
phi3 = -0.35
phi5 = 0.25
phi7 = -0.15

i = (
    I1*np.sin(w1*t + phi1)
    + I3*np.sin(3*w1*t + phi3)
    + I5*np.sin(5*w1*t + phi5)
    + I7*np.sin(7*w1*t + phi7)
)

## 3. Forma temporal de la corriente

La deformación visible respecto de una sinusoidal pura sugiere ya la presencia de armónicos superiores.

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(t*1000, i)
plt.xlabel("t [ms]")
plt.ylabel("i(t) [A]")
plt.title("Corriente periódica no sinusoidal")
plt.grid(True, alpha=0.3)
plt.show()

## 4. Un periodo aislado

Para apreciar mejor la estructura de la forma de onda, conviene mirar un solo periodo.

In [ ]:
one_period = samples_per_period
t1 = t[:one_period]
i1 = i[:one_period]

plt.figure(figsize=(10, 4.5))
plt.plot(t1*1000, i1)
plt.xlabel("t [ms]")
plt.ylabel("i(t) [A]")
plt.title("Un periodo de la corriente")
plt.grid(True, alpha=0.3)
plt.show()

## 5. FFT y espectro de amplitudes

Calculamos la transformada discreta de Fourier para extraer el contenido espectral de la señal.

In [ ]:
dt = t[1] - t[0]
freqs = rfftfreq(N, d=dt)
If = rfft(i)

amps = 2*np.abs(If)/N
amps[0] = np.abs(If[0])/N

harmonic_order = freqs / f1

In [ ]:
max_order = 15
mask = harmonic_order <= max_order

plt.figure(figsize=(10, 4.5))
plt.stem(harmonic_order[mask], amps[mask], basefmt=" ")
plt.xlabel("Orden armónico")
plt.ylabel("Amplitud [A]")
plt.title("Espectro de amplitudes")
plt.grid(True, alpha=0.3)
plt.show()

## 6. Extraer amplitudes por armónico

Como la señal fue construida armónicamente, esperamos recuperar amplitudes cercanas a las prescritas.

In [ ]:
def get_harmonic_amplitude(k, harmonic_order, amps):
    idx = np.argmin(np.abs(harmonic_order - k))
    return harmonic_order[idx], amps[idx]

orders_to_check = [1, 3, 5, 7, 9, 11]
rows = []
for k in orders_to_check:
    hk, ak = get_harmonic_amplitude(k, harmonic_order, amps)
    rows.append((k, hk, ak))

print("k | orden detectado | amplitud [A]")
print("-" * 38)
for k, hk, ak in rows:
    print(f"{k:1d} | {hk:14.6f} | {ak:12.6f}")

## 7. Cálculo de THD

Usamos la definición clásica:

$$
\mathrm{THD} = \frac{\sqrt{\sum_{h\ge 2} I_h^2}}{I_1}.
$$

Aquí calcularemos el THD a partir de las amplitudes armónicas recuperadas por FFT.

In [ ]:
max_h_for_thd = 25
Ih = {}

for k in range(1, max_h_for_thd + 1):
    _, ak = get_harmonic_amplitude(k, harmonic_order, amps)
    Ih[k] = ak

I1_est = Ih[1]
thd = np.sqrt(sum(Ih[k]**2 for k in Ih if k >= 2)) / I1_est

print(f"I1 estimada = {I1_est:.6f} A")
print(f"THD = {100*thd:.4f} %")

## 8. Reconstrucción truncada

Ahora comparamos la señal original con reconstrucciones que retienen:
- solo la fundamental,
- hasta el tercer armónico,
- hasta el séptimo armónico.

In [ ]:
def reconstruct_from_harmonics(signal, t, f1, max_order):
    N = len(signal)
    dt = t[1] - t[0]
    freqs = rfftfreq(N, d=dt)
    Sf = rfft(signal)
    orders = freqs / f1

    Sfiltered = np.zeros_like(Sf, dtype=complex)
    for idx, order in enumerate(orders):
        k = int(np.round(order))
        if np.isclose(order, k, atol=1e-6) and k <= max_order:
            Sfiltered[idx] = Sf[idx]
    recon = np.fft.irfft(Sfiltered, n=N)
    return recon

i_fund = reconstruct_from_harmonics(i1, t1, f1, max_order=1)
i_3 = reconstruct_from_harmonics(i1, t1, f1, max_order=3)
i_7 = reconstruct_from_harmonics(i1, t1, f1, max_order=7)

In [ ]:
plt.figure(figsize=(10, 4.8))
plt.plot(t1*1000, i1, linewidth=2.0, label="Original")
plt.plot(t1*1000, i_fund, "--", label="Solo fundamental")
plt.plot(t1*1000, i_3, "--", label="Hasta 3er armónico")
plt.plot(t1*1000, i_7, "--", label="Hasta 7mo armónico")
plt.xlabel("t [ms]")
plt.ylabel("i(t) [A]")
plt.title("Reconstrucción truncada de la corriente")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 9. Error RMS de reconstrucción

Medimos qué tan cerca queda cada reconstrucción truncada respecto de la señal original.

In [ ]:
def rms_error(x, y):
    return np.sqrt(np.mean((x - y)**2))

print("Error RMS solo fundamental :", rms_error(i1, i_fund))
print("Error RMS hasta 3er armónico:", rms_error(i1, i_3))
print("Error RMS hasta 7mo armónico:", rms_error(i1, i_7))

## 10. Corriente y tensión senoidal: potencia instantánea

Para dar un pequeño paso hacia interpretación eléctrica, supongamos una tensión puramente senoidal:

$$
v(t)=V_1\sin(\omega_1 t).
$$

Entonces la potencia instantánea será

$$
p(t)=v(t)i(t),
$$

y mostrará oscilaciones adicionales ligadas a la distorsión de corriente.

In [ ]:
V1 = 127.0*np.sqrt(2)   # tensión pico equivalente a 127 V rms
v = V1*np.sin(w1*t + 0.0)
p = v * i

plt.figure(figsize=(10, 4.5))
plt.plot(t1*1000, v[:one_period], label="v(t)")
plt.plot(t1*1000, i1, label="i(t)")
plt.xlabel("t [ms]")
plt.title("Tensión senoidal y corriente distorsionada")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(t1*1000, p[:one_period])
plt.xlabel("t [ms]")
plt.ylabel("p(t)")
plt.title("Potencia instantánea en un periodo")
plt.grid(True, alpha=0.3)
plt.show()

## 11. Comentario conceptual

Esta exploración muestra varias ideas importantes:

- una corriente no sinusoidal puede leerse como una **estructura armónica organizada**;
- el espectro ofrece una descripción compacta de esa estructura;
- el **THD** resume parcialmente el grado de distorsión;
- la reconstrucción truncada permite evaluar cuánto aporta cada conjunto de armónicos;
- incluso con tensión senoidal, una corriente distorsionada modifica la forma de la potencia instantánea.

Este tipo de lectura prepara el terreno para modelos más ricos en los que una fuente, una línea, un transformador y una carga no lineal interactúan dentro de un marco armónico computacional.

## 12. Exploraciones sugeridas

1. Cambia las amplitudes `I3`, `I5`, `I7` y observa cómo cambia el THD.
2. Introduce armónicos pares y compara el resultado.
3. Cambia la fase de algunos armónicos y observa la deformación temporal.
4. Construye una señal con mayor contenido en alta frecuencia.
5. Explora cómo cambia la potencia instantánea al modificar la distorsión de corriente.

## 13. Hacia un laboratorio armónico mínimo

Este notebook puede entenderse como la primera celda de un pequeño laboratorio armónico en Python/Jupyter.  
A partir de aquí se puede avanzar hacia una biblioteca mínima que incluya:

- representación y reconstrucción de señales periódicas,
- extracción de coeficientes de Fourier,
- métricas de distorsión,
- y, posteriormente, modelos sencillos del tipo
  **fuente → transformador → línea → carga no lineal**.